In [1]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator



/tmp/ipykernel_512/2110009738.py:12: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [2]:
os.environ["HF_TOKEN"] = input()

In [3]:
#dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")
dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")


README.md:   0%|          | 0.00/622 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.20MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  118kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/574 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/577 [00:00<?, ? examples/s]

In [10]:
traind_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, val_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['query'])

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['index', 'doc_id', 'link', 'title', 'text'],
    num_rows: 1151
})

In [12]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [13]:
# FT_MODEL_ID = "pameydorke/arcanum-cross-platform-retriever"
FT_MODEL_ID = "pameydorke/arcanumchat-retriever-bge-m3"
BASE_MODEL_ID = "BAAI/bge-m3"

In [14]:
base_model = SentenceTransformer('BAAI/bge-m3')
ft_model = SentenceTransformer('pameydorke/arcanum-cross-platform-retriever')

base_model.eval()
ft_model.eval()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [33]:
queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["link"]
    doc_text = example["text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        doc_id = example["doc_id"]
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
vague_counter = 0
for idx, example in enumerate(test_ds):
    # if example['is_vague']:
    #     vague_counter += 1
    #     continue

    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)
print("Skipped VAGUE:", vague_counter)

Skipped: 2
Skipped VAGUE: 0


In [34]:
base_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)
ft_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)

base_results = base_evaluator(base_model)
ft_results = ft_evaluator(ft_model)

In [35]:
base_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_accuracy@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_precision@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_precision@3': 0.26140350877192975,
 'arcanum-retriever-test_cosine_precision@5': 0.16947368421052625,
 'arcanum-retriever-test_cosine_precision@10': 0.08684210526315787,
 'arcanum-retriever-test_cosine_recall@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_recall@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_recall@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_recall@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7664273704408339,
 'arcanum-retriever-test_cosine_mrr@10': 0.7331766917293234,
 'arcanum-retriever-test_cosine_map@100': 0.7376623485506122}

In [36]:
ft_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6263157894736842,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_accuracy@10': 0.8947368421052632,
 'arcanum-retriever-test_cosine_precision@1': 0.6263157894736842,
 'arcanum-retriever-test_cosine_precision@3': 0.26140350877192975,
 'arcanum-retriever-test_cosine_precision@5': 0.16947368421052625,
 'arcanum-retriever-test_cosine_precision@10': 0.0894736842105263,
 'arcanum-retriever-test_cosine_recall@1': 0.6263157894736842,
 'arcanum-retriever-test_cosine_recall@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_recall@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_recall@10': 0.8947368421052632,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7638678937223286,
 'arcanum-retriever-test_cosine_mrr@10': 0.72171679197995,
 'arcanum-retriever-test_cosine_map@100': 0.7259463823721104}

In [37]:
import math
import torch
import numpy as np
import pandas as pd
from sentence_transformers import util


def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [38]:
def per_query_metrics(model, queries, corpus, relevant_docs, k_values=(1, 5, 10)):
    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    # (num_queries, num_corpus)
    sim = util.cos_sim(q_emb, c_emb)

    results = {}

    for i, qid in enumerate(qids):
        scores = sim[i]
        ranked_idx = torch.argsort(scores, descending=True).tolist()
        ranked_cids = [cids[j] for j in ranked_idx]

        rel = relevant_docs.get(qid, set())
        metrics = {}

        for k in k_values:
            topk = ranked_cids[:k]

            # recall@k
            hits = sum(1 for d in topk if d in rel)
            metrics[f"recall@{k}"] = hits / len(rel) if rel else 0.0

            # ndcg@k
            dcg = 0.0
            for rank, d in enumerate(topk, start=1):
                if d in rel:
                    dcg += 1.0 / math.log2(rank + 1)

            idcg = sum(
                1.0 / math.log2(r + 1)
                for r in range(1, min(len(rel), k) + 1)
            )
            metrics[f"ndcg@{k}"] = dcg / idcg if idcg > 0 else 0.0

        # MRR
        rr = 0.0
        for rank, d in enumerate(ranked_cids, start=1):
            if d in rel:
                rr = 1.0 / rank
                break
        metrics["mrr"] = rr

        results[qid] = metrics

    return results

In [39]:
base_per_q = per_query_metrics(base_model, queries, corpus, relevant_docs)

In [40]:
ft_per_q   = per_query_metrics(ft_model,   queries, corpus, relevant_docs)

In [41]:
metric = "recall@10"

base = np.array([base_per_q[q][metric] for q in queries])
ft   = np.array([ft_per_q[q][metric]   for q in queries])
delta = ft - base

print("base mean:", base.mean())
print("ft mean:  ", ft.mean())
print("mean delta:", delta.mean())
print("improved:", (delta > 0).sum())
print("worse:   ", (delta < 0).sum())
print("same:    ", (delta == 0).sum())

rng = np.random.default_rng(0)
n = len(delta)

boot = np.array([
    delta[rng.integers(0, n, n)].mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean delta = {delta.mean():.4f}", "\n")
print(f"95% CI     = [{lo:.4f}, {hi:.4f}]")

base mean: 0.868421052631579
ft mean:   0.8947368421052632
mean delta: 0.02631578947368421
improved: 7
worse:    2
same:     181
mean delta = 0.0263 

95% CI     = [0.0000, 0.0579]


In [42]:
metric = "ndcg@10"

base = np.array([base_per_q[q][metric] for q in queries])
ft   = np.array([ft_per_q[q][metric]   for q in queries])
delta = ft - base

print("base mean:", base.mean())
print("ft mean:  ", ft.mean())
print("mean delta:", delta.mean())
print("improved:", (delta > 0).sum())
print("worse:   ", (delta < 0).sum())
print("same:    ", (delta == 0).sum())

rng = np.random.default_rng(0)
n = len(delta)

boot = np.array([
    delta[rng.integers(0, n, n)].mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean delta = {delta.mean():.4f}", "\n")
print(f"95% CI     = [{lo:.4f}, {hi:.4f}]")

base mean: 0.7664273704408339
ft mean:   0.7638678937223286
mean delta: -0.002559476718505304
improved: 25
worse:    26
same:     139
mean delta = -0.0026 

95% CI     = [-0.0285, 0.0236]


In [43]:
rows = []
for qid in queries:
    row = {"qid": qid, "query": queries[qid]}
    for k, v in base_per_q[qid].items():
        row[f"base_{k}"] = v
    for k, v in ft_per_q[qid].items():
        row[f"ft_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows)
df["delta_ndcg@10"] = df["ft_ndcg@10"] - df["base_ndcg@10"]
df = df.sort_values("delta_ndcg@10")

# Queries where fine-tuning hurt most
print(df[["qid", "query", "base_ndcg@10", "ft_ndcg@10", "delta_ndcg@10"]].head(20))

           qid                                            query  base_ndcg@10  \
153  query_155     UAP install different version mid-playthrouh      1.000000   
33    query_33                                 build calculator      1.000000   
32    query_32                           character build planer      1.000000   
185  query_187                         GOG version on modern PC      1.000000   
182  query_184       return sold quest item by prototype id UAP      1.000000   
61    query_63                                  ancesctral swrd      1.000000   
10    query_10                 Arcane Sword dungeon coordinates      1.000000   
9      query_9  can't play game is small and in top left corner      1.000000   
99   query_101           Tarnat guards attack after killing npc      1.000000   
34    query_34                      late game area game crashes      1.000000   
162  query_164                               sence hiiden spell      1.000000   
184  query_186              

In [44]:
# Build a list of (qid, base, ft, delta) for recall@5
rows = []
for qid in queries:
    b = base_per_q[qid]["recall@10"]
    f = ft_per_q[qid]["recall@10"]
    rows.append((qid, b, f, f - b))

# Keep only the ones that got worse
worse = [r for r in rows if r[3] < 0]
worse.sort(key=lambda r: r[3])  # most negative first

print(f"Total worse on recall@10: {len(worse)}")
print("=" * 80)

for qid, b, f, d in worse:
    # print(f"qid:     {qid}")
    print(f"query:   {queries[qid]}")
    # print(f"base:    {b:.3f}   ft: {f:.3f}   delta: {d:+.3f}")

    rel_docs = relevant_docs[qid]
    # print(f"relevant doc ids: {rel_docs}")

    # Show the text of the relevant docs (truncated)
    for doc_id in rel_docs:
        text = corpus[doc_id]
        # print(f"  [{doc_id}] {text[:300]}...")

    # If you want to see where the fine-tuned model ranked the gold doc,
    # you need the ranked list. See helper below.
    print("-" * 80)

Total worse on recall@10: 2
query:   find elves
--------------------------------------------------------------------------------
query:   UAP
--------------------------------------------------------------------------------
